# Spanish Transcritption

## Installations

In [ ]:
!pip install python-Levenshtein

In [ ]:
!pip install pyannote.audio==3.1.1 --upgrade

In [ ]:
!pip install faster_whisper

In [ ]:
!pip install pyannote.audio


In [ ]:
!pip install noisereduce

In [ ]:
!pip install python-Levenshtein openpyxl -q

## Imports


In [ ]:
import numpy as np
import torch
import noisereduce as nr
import soundfile as sf
from faster_whisper import WhisperModel
from pydub import AudioSegment
import tempfile, os
import Levenshtein
import re
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Preprocesssing

### Load and Resample

In [ ]:
def load_and_resample(path: str, target_sr: int = 16000):
    audio = AudioSegment.from_file(path)
    audio = audio.set_channels(1).set_frame_rate(target_sr)
    samples = np.array(audio.get_array_of_samples()).astype(np.float32)
    samples /= np.iinfo(audio.array_type).max   # normalize to [-1, 1]
    return samples, target_sr

### VAD

In [ ]:
def run_vad(samples: np.ndarray, sr: int, thr: float):
    model, utils = torch.hub.load(
        repo_or_dir="snakers4/silero-vad",
        model="silero_vad",
        force_reload=False
    )
    (get_speech_timestamps, _, read_audio, *_) = utils

    audio_tensor = torch.FloatTensor(samples)
    speech_timestamps = get_speech_timestamps(
        audio_tensor, model,
        sampling_rate=sr,
        threshold=thr,
        min_speech_duration_ms=250,
        min_silence_duration_ms=300,
    )
    segments = [(t["start"] / sr, t["end"] / sr) for t in speech_timestamps]
    print(f"[VAD] Found {len(segments)} speech segments")
    return segments

### Add Padding

In [ ]:
def add_padding(segments, sr, pad_ms=200):
    padded = []
    pad = int(pad_ms * sr / 1000)

    for s, e in segments:
        new_s = max(0, s - pad/sr)
        new_e = e + pad/sr
        padded.append((new_s, new_e))

    return padded

### Concatenate only the VAD speech chunks

In [ ]:
def extract_speech_audio(samples: np.ndarray, sr: int, segments):
    chunks = [samples[int(s * sr): int(e * sr)] for s, e in segments]
    speech_audio = np.concatenate(chunks) if chunks else samples
    print(f"[VAD] Kept {len(speech_audio)/sr:.1f}s of speech "
          f"from {len(samples)/sr:.1f}s total")
    return speech_audio

### Noise Reduction

In [ ]:
def reduce_noise(samples: np.ndarray, sr: int, pd: float):
    """Spectral noise reduction (noisereduce)."""
    denoised = nr.reduce_noise(y=samples, sr=sr, stationary=False, prop_decrease=pd)
    print("[Noise Reduction] Done")
    return denoised

### Normalize Loudness

In [ ]:
def normalize_audio(samples):
    max_val = np.max(np.abs(samples))
    if max_val > 0:
        samples = samples / max_val
    return samples

In [ ]:
def remove_dc_offset(samples):
    return samples - np.mean(samples)

### Merge Vad Segments before

In [ ]:
def merge_vad_segments(segments, max_gap=0.5):
    merged = []
    current_start, current_end = segments[0]

    for s, e in segments[1:]:
        if s - current_end <= max_gap:
            current_end = e
        else:
            merged.append((current_start, current_end))
            current_start, current_end = s, e

    merged.append((current_start, current_end))
    return merged

## Transcription

### Transcribe

In [ ]:
def transcribe(samples: np.ndarray, sr: int, speech_segments: list):
    model = WhisperModel("medium", device="cuda", compute_type="float16")

    results = []
    for i, (start, end) in enumerate(speech_segments):
        chunk = samples[int(start * sr): int(end * sr)]

        # Skip very short chunks
        if len(chunk) / sr < 0.5:
            continue

        segments, info = model.transcribe(
            chunk,
            language= "es",
            beam_size=5,
            vad_filter=False,
        )

        for seg in segments:
            results.append({
                "start": start + seg.start,   # offset by chunk start
                "end":   start + seg.end,
                "text":  seg.text.strip(),
                "lang":  info.language,
            })
            print(f"  [{start + seg.start:.1f}s → start + {seg.end:.1f}s] "
                  f"[{info.language}] {seg.text.strip()}")

    return results

### Merging

In [ ]:
def merge_until_target(results, target_count=30):
    current = results.copy()

    while len(current) > target_count:
        # Find shortest segment index
        lengths = [len(r["text"].split()) for r in current]
        idx = lengths.index(min(lengths))

        # If it's last element, merge with previous
        if idx == len(current) - 1:
            merge_idx = idx - 1
        else:
            merge_idx = idx

        # Merge current[merge_idx] and current[merge_idx + 1]
        merged = {
            "start": current[merge_idx]["start"],
            "end": current[merge_idx + 1]["end"],
            "text": current[merge_idx]["text"] + " " + current[merge_idx + 1]["text"]
        }

        # Replace the two with merged
        current = (
            current[:merge_idx] +
            [merged] +
            current[merge_idx + 2:]
        )

    return current

## Post Processing

### Stimulus List

In [ ]:
STIMULI = [
    (1,  "Quiero cortarme el pelo"),
    (2,  "El libro está en la mesa"),
    (3,  "El carro lo tiene Pedro"),
    (4,  "El se ducha cada mañana"),
    (5,  "¿Qué dice usted que va a hacer hoy?"),
    (6,  "Dudo que sepa manejar muy bien"),
    (7,  "Las calles de esta ciudad son muy anchas"),
    (8,  "Puede que llueva mañana todo el día"),
    (9,  "Las casas son muy bonitas pero caras"),
    (10, "Me gustan las películas que acaban bien"),
    (11, "El chico con el que yo salgo es español"),
    (12, "Después de cenar me fui a dormir tranquilo"),
    (13, "Quiero una casa en la que vivan mis animales"),
    (14, "A nosotros nos fascinan las fiestas grandiosas"),
    (15, "Ella sólo bebe cerveza y no come nada"),
    (16, "Me gustaría que el precio de las casas bajara"),
    (17, "Cruza a la derecha y después sigue todo recto"),
    (18, "Ella ha terminado de pintar su apartamento"),
    (19, "Me gustaría que empezara a hacer más calor pronto"),
    (20, "El niño al que se le murió el gato está triste"),
    (21, "Una amiga mía cuida a los niños de mi vecino"),
    (22, "El gato que era negro fue perseguido por el perro"),
    (23, "Antes de poder salir él tiene que limpiar su cuarto"),
    (24, "La cantidad de personas que fuman ha disminuido"),
    (25, "Después de llegar a casa del trabajo tomé la cena"),
    (26, "El ladrón al que atrapó la policía era famoso"),
    (27, "Le pedí a un amigo que me ayudara con la tarea"),
    (28, "El examen no fue tan difícil como me habían dicho"),
    (29, "¿Serías tan amable de darme el libro que está en la mesa?"),
    (30, "Hay mucha gente que no toma nada para el desayuno"),
]


# ── Drop this into your pipeline after transcribe() ──────────────────────────


### Normalizing Stimulus

In [ ]:
def normalize(text):
    text = text.replace("<pause>", "")
    return re.sub(r'[^\w\s]', '', text.lower().strip())

### Merging Nearby fragments

In [ ]:
## merging 2 transcriptions if they have a gap of less than 2 seconds, and adding a <pause> token.
def merge_nearby_fragments(results, gap_threshold_s=2.0):
    if not results:
        return results

    merged = [results[0].copy()]
    for curr in results[1:]:
        prev = merged[-1]
        gap = curr["start"] - prev["end"]
        if gap <= gap_threshold_s:
            prev["end"]  = curr["end"]
            prev["text"] = prev["text"].rstrip() + " <pause> " + curr["text"].lstrip()  # ← only change
        else:
            merged.append(curr.copy())

    print(f"[Merge] {len(results)} segments → {len(merged)} after merging (gap ≤ {gap_threshold_s}s)")
    return merged




### Matching ASR to Stimuli

In [ ]:
### greedy algorithm that matches stuff
def match_asr_to_stimuli(results, stimuli, max_dist=20):
    used = set()
    matched = []

    for num, stim in stimuli:
        best_score = float('inf')
        best_text  = ""
        best_i     = None

        for i, r in enumerate(results):
            if i in used:
                continue
            dist = Levenshtein.distance(normalize(r["text"]), normalize(stim))
            if dist < best_score:
                best_score = dist
                best_text  = r["text"].strip()
                best_i     = i

        if best_i is not None and best_score <= max_dist:   # ← threshold check
            used.add(best_i)
            transcription = best_text
        else:
            transcription = "[not produced]"               # ← flag instead of wrong match

        matched.append({
            "sentence":      num,
            "stimulus":      stim,
            "transcription": transcription,
        })
        print(f"  {num:>2}. [{stim}]\n      → {transcription}  (dist={best_score})\n")

    return matched



### Export

In [ ]:
### export
def export_to_excel(matched, output_path):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Transcriptions"

    thin = Side(style="thin", color="CCCCCC")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # Header
    for col, label in enumerate(["Sentence", "Stimulus", "Transcription"], 1):
        cell = ws.cell(row=1, column=col, value=label)
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=11)
        cell.fill      = PatternFill("solid", start_color="4472C4")
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border    = border

    # Data rows
    for r in matched:
        ws.append([r["sentence"], r["stimulus"], r["transcription"]])
        for col in range(1, 4):
            cell = ws.cell(row=ws.max_row, column=col)
            cell.alignment = Alignment(wrap_text=True, vertical="top")
            cell.border    = border
            cell.font      = Font(name="Arial", size=11)

    ws.column_dimensions["A"].width = 10
    ws.column_dimensions["B"].width = 52
    ws.column_dimensions["C"].width = 52
    ws.row_dimensions[1].height     = 20

    wb.save(output_path)
    print(f"\n[Export] Saved → {output_path}")

## Pipeline

### Config

In [ ]:
# ─── CONFIG ───────────────────────────────────────────────────────────────────
AUDIO_PATH     = "/content/drive/MyDrive/spanish/038010_EIT-2A.mp3"   # any format
TARGET_SR      = 16000              # Whisper expects 16kHz
HF_TOKEN       = ""         # for pyannote VAD
TRANSCRIBE_LANG = None                    # None = auto-detect; "es" = force Spanish
# ──────────────────────────────────────────────────────────────────────────────

### End-to-End

In [ ]:
if __name__ == "__main__":
    audio, sr = load_and_resample(AUDIO_PATH, TARGET_SR)

    # Light cleaning
    audio = remove_dc_offset(audio)
    audio = normalize_audio(audio)

    # VAD
    speech_segments = run_vad(audio, sr, thr=0.35)

    # Padding
    speech_segments = add_padding(speech_segments, sr, pad_ms=200)

    # Merge segments (context preservation)
    speech_segments = merge_vad_segments(speech_segments, max_gap=0.5)

    # Optional denoise
    USE_DENOISE = False
    if USE_DENOISE:
        audio = reduce_noise(audio, sr, pd=0.5)

    # Transcribe
    results = transcribe(audio, sr, speech_segments)

    print("\n─── RAW SEGMENTS ───")
    for r in results:
        print(r["text"])

    # print("\n─── STEP 1: MERGE NEARBY FRAGMENTS ───")
    # merged_segments = merge_nearby_fragments(results, gap_threshold_s=2.0)

    print("\n─── STEP 2: MATCH TO STIMULI ───")
    matched = match_asr_to_stimuli(merged_segments, STIMULI)

    OUTPUT_PATH = "/content/drive/MyDrive/spanish/transcription_output.xlsx"
    export_to_excel(matched, OUTPUT_PATH)



Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


[VAD] Found 53 speech segments
  [73.9s → start + 2.0s] [es] nos fuimos al parque.
  [81.4s → start + 2.0s] [es] La llamaré mañana.
  [88.8s → start + 2.0s] [es] Puedes comprar carne en la tienda de butcher.
  [97.2s → start + 2.8s] [es] Mi hermano recibió un nuevo computador.
  [107.5s → start + 3.0s] [es] A veces toman a su papá para un paseo en el parque.
  [118.9s → start + 3.0s] [es] Voy a jugar a la voleybal en el gimnasio que os he dicho.
  [165.1s → start + 2.0s] [es] quiero cortarme el pelo
  [172.4s → start + 2.0s] [es] El libro está en la mesa.
  [179.8s → start + 2.0s] [es] El caro lo tiene pelo.
  [187.9s → start + 2.0s] [es] ¡Ese duché cada mañana!
  [197.0s → start + 2.8s] [es] ¿Qué dices ustedes que van a hacer hoy?
  [206.1s → start + 3.0s] [es] Duro que sepa manejar muy bien.
  [215.9s → start + 3.0s] [es] Las calles de esta ciudad son muy anchas.
  [227.3s → start + 3.0s] [es] Puede que lleve mañana toda el día.
  [238.6s → start + 5.0s] [es] Las casas son muy bonita